# Silver Validation

Lightweight validation for source-aligned Silver outputs. This notebook reads artifacts generated by `src/` jobs and does not define pipeline logic.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "src" / "processing").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "src" / "processing").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "projects" / "03-retail-revenue-analytics" / "src" / "processing").exists():
    PROJECT_ROOT = cwd / "projects" / "03-retail-revenue-analytics"
else:
    raise RuntimeError("Launch this notebook from the project directory, notebook directory, or repository root.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from processing.silver.config import get_silver_tables_dir, get_silver_metadata_dir

silver_tables_dir = get_silver_tables_dir()
silver_metadata_dir = get_silver_metadata_dir()
silver_tables_dir, silver_metadata_dir

## Available Silver Tables

In [ ]:
silver_files = sorted(silver_tables_dir.glob("*.csv"))
silver_files

In [ ]:
import pandas as pd

tables = {path.stem: pd.read_csv(path) for path in silver_files}
{name: dataframe.shape for name, dataframe in tables.items()}

## Table Previews

In [ ]:
for name, dataframe in tables.items():
    print(f"\n{name}: {dataframe.shape[0]} rows x {dataframe.shape[1]} columns")
    display(dataframe.head())

## Columns and Null Counts

In [ ]:
for name, dataframe in tables.items():
    print(f"\n{name}")
    print(list(dataframe.columns))
    display(dataframe.isna().sum().sort_values(ascending=False).head(10))

## Simple Grain Checks

These checks are exploratory. Stable data quality tests should live outside notebooks once contracts are finalized.

In [ ]:
checks = {}
if "orders" in tables:
    checks["orders_duplicate_order_id"] = int(tables["orders"].duplicated("order_id").sum())
if "order_items" in tables:
    checks["order_items_duplicate_order_item"] = int(tables["order_items"].duplicated(["order_id", "order_item_id"]).sum())
if "products" in tables:
    checks["products_duplicate_product_id"] = int(tables["products"].duplicated("product_id").sum())
if "customers" in tables:
    checks["customers_duplicate_customer_id"] = int(tables["customers"].duplicated("customer_id").sum())
if "sellers" in tables:
    checks["sellers_duplicate_seller_id"] = int(tables["sellers"].duplicated("seller_id").sum())
checks